In [2]:
import os
import pandas as pd

from scripts.utils import get_mend_df, rep_mend_analysis, mend_f1_table, get_thresh_score, add_cm_col, export_cm_examples
from src.utils import load_env, set_seed,load_json, get_logger
from src.experiment_config import ExperimentConfig

/Users/alexleto/projects/metaphor-detector/env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# load file into dataframe
llm_vc_data = load_json("/Users/alexleto/projects/metaphor-detector/results/llm_ann/llm_source_classifier/source_verb_classifications.json")["data"]
llm_vc_dicts = [ 
    {   "id": str(key), "doc_id": str(key).split("_")[0], "sent_id": str(key).split("_")[1], "path_id": str(key).split("_")[2], \
        "text": entry["text"], \
        "llm_met_class": entry["annotation"]["classification"], "llm_explanation": entry["annotation"]["explanation"] \
    }
    for key, entry in llm_vc_data.items()
]
llm_vc_df = pd.DataFrame.from_dict(llm_vc_dicts)

In [4]:
# get stats
class_grouped = llm_vc_df.groupby("llm_met_class").agg(count = ("doc_id", "count"))
class_grouped

,count
llm_met_class,
Literal,493
Metaphorical,4392


In [5]:
# randomly print 10 literal
n = 10
lit_df = llm_vc_df[llm_vc_df["llm_met_class"] == "Literal"]
ran_lit_rows = lit_df.sample(n=n)
for _, row in ran_lit_rows.iterrows():
    print(row["text"])
    print(row["llm_explanation"])
    print()

Specified verb: 'Offer'
 Sentence: 'Offer free college...'
The verb 'offer' is used to describe a physical action of presenting or making available something, in this case, free college education. The sentence focuses on the provision of educational resources without an additional charge, which aligns with the literal meaning of offering. There is no personification or animalization of objects present in the sentence, and the verb's usage does not imply a conceptual equivalence to any non-physical action.

Specified verb: 'died'
 Sentence: 'Five immigrant children have died in US custody since December.'
The verb 'died' is used to describe a physical event, specifically the demise of living individuals. In this context, it refers to the termination of life due to natural or unnatural causes, which aligns with its literal meaning. The phrase does not attribute human-like qualities to an inanimate object or use personification, and there are no clear indications that the verb has been ex

In [6]:
# randomly print 10 literal
n = 10
met_df = llm_vc_df[llm_vc_df["llm_met_class"] == "Metaphorical"]
ran_met_rows = met_df.sample(n=n)
print("METAPHORICAL EXAMPLES\n")
for _, row in ran_met_rows.iterrows():
    print(row["text"])
    print(row["llm_explanation"])
    print()

METAPHORICAL EXAMPLES

Specified verb: 'give'
 Sentence: 'They should get what they give!'
In this sentence, the verb 'give' is used metaphorically to convey the idea that individuals will receive the same kind of treatment or consequences as they have previously provided. This usage involves a transfer of meaning from its literal sense (physically providing something) to an idiomatic expression that conveys reciprocity and consequence. The focus here is not on physical action, but rather on the concept of cause-and-effect relationships and retribution.

Specified verb: 'suffer'
 Sentence: 'GOP still playing games in immigration while people suffer the consequences of their partisan politics games.'
The verb 'suffer' is used metaphorically to describe the negative impact of the GOP's actions, rather than its literal meaning of experiencing physical pain or hardship. In this context, 'suffering' refers to the consequences of their partisan politics being felt by people, which is a figur